# DenseNet169 for CBIS-DDSM Mammogram Classification (V1)

## Overview

This notebook implements DenseNet169 with optimizations derived from peer-reviewed literature for breast cancer classification using the CBIS-DDSM dataset.

### Research Foundation

**Primary References:**
1. Talukder et al. (2025). "An improved XAI-based DenseNet model for breast cancer detection." *Results in Engineering*.
2. Rasheed et al. (2025). "Improved brain tumor classification through DenseNet121 based transfer learning." *Discover Oncology*.

### Key Findings from Literature

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Architecture | DenseNet169 | Outperformed DenseNet121 by 0.25% (Talukder et al.) |
| Input Resolution | 256×256 | Optimal for mammographic feature extraction |
| Learning Rate | 1e-4 | Consistent across both studies |
| Optimizer | Adamax | Superior convergence for medical imaging |
| Preprocessing | CLAHE + Sharpening | +1.25% accuracy improvement |

### Computational Requirements
- GPU: NVIDIA A100 (40GB VRAM) recommended
- Dataset: CBIS-DDSM ROI images

In [ ]:
# =============================================================================
# Environment Setup
# =============================================================================

import os
import sys
import warnings
import hashlib
import pickle
warnings.filterwarnings('ignore')

# GPU verification
import subprocess
try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']
    ).decode('utf-8').strip()
    print(f"GPU: {gpu_info}")
except:
    print("Warning: No GPU detected")

# Google Colab environment detection
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Reproducibility
RANDOM_SEED = 1024

import random
import numpy as np
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Environment: {'Colab' if IN_COLAB else 'Local'} | Seed: {RANDOM_SEED}")

In [ ]:
# =============================================================================
# Dependencies
# =============================================================================

!pip install -q albumentations scikit-learn matplotlib seaborn tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.applications import DenseNet169
from tensorflow.keras.optimizers import Adamax, Adam, RMSprop
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve, 
    accuracy_score, precision_recall_fscore_support
)
import json
import pickle
import hashlib
from datetime import datetime
from collections import Counter
import gc

tf.random.set_seed(RANDOM_SEED)

# Mixed precision for A100
try:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
except:
    pass

# GPU memory configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow {tf.__version__} | GPUs: {len(gpus)}")

In [ ]:
# =============================================================================
# Data Path Verification
# =============================================================================

from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/CBIS-DDSM-data") if IN_COLAB else Path("/home/tars/GoogleDrive/CBIS-DDSM-data")

if DATA_ROOT.exists():
    print(f"Data root: {DATA_ROOT}")
    
    # Check for preprocessed cache (primary data source)
    cache_path = DATA_ROOT / "preprocessed_cache_roi"
    if cache_path.exists():
        cache_files = list(cache_path.glob("*.npz"))
        print(f"Cache directory: {cache_path}")
        print(f"  Cache files: {[f.name for f in cache_files]}")
    else:
        print(f"WARNING: Cache directory not found: {cache_path}")
else:
    print(f"ERROR: Data root not found: {DATA_ROOT}")

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

class Config:
    """Hyperparameters derived from literature review."""
    
    # Model
    MODEL_NAME = "DenseNet169_V1_ROI"
    VERSION = "1.0.0"
    
    # Input
    IMG_SIZE = 256
    IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
    
    # Training (Talukder et al., Rasheed et al.)
    LEARNING_RATE = 1e-4
    OPTIMIZER = "adamax"
    BATCH_SIZE = 32
    EPOCHS_STAGE1 = 15  # Classifier warm-up
    EPOCHS_STAGE2 = 25  # Progressive unfreezing
    EPOCHS_STAGE3 = 15  # Fine-tuning
    
    # Regularization
    DROPOUT_FINETUNE = 0.002  # Talukder et al.
    DROPOUT_GENERAL = 0.2     # Rasheed et al.
    SPATIAL_DROPOUT = 0.1
    L2_REG = 1e-4
    LABEL_SMOOTHING = 0.1
    
    # Preprocessing (applied in cached data)
    CLAHE_CLIP_LIMIT = 2.5
    CLAHE_TILE_SIZE = (8, 8)
    USE_SHARPENING = True
    USE_MEDIAN_FILTER = True
    MEDIAN_KERNEL = 3
    USE_BILATERAL = True
    
    # Augmentation (Talukder et al.)
    ROTATION_RANGE = 20
    SHIFT_RANGE = 0.3
    ZOOM_RANGE = 0.1
    HORIZONTAL_FLIP = True
    
    # Ensemble
    NUM_ENSEMBLE = 3
    TTA_AUGMENTATIONS = 8
    
    # Paths
    DATA_ROOT = Path("/content/drive/MyDrive/CBIS-DDSM-data") if IN_COLAB else Path("/home/tars/GoogleDrive/CBIS-DDSM-data")
    CACHE_DIR = DATA_ROOT / "preprocessed_cache_roi"
    CHECKPOINT_DIR = DATA_ROOT / "checkpoints_densenet169_v1"
    RESULTS_DIR = DATA_ROOT / "results_densenet169_v1"
    
    CLASS_NAMES = ['BENIGN', 'MALIGNANT']
    SEED = RANDOM_SEED


def verify_cache_directory(config):
    """Verify preprocessed cache directory exists."""
    cache_dir = config.CACHE_DIR
    required_files = ['train_cache.npz', 'val_cache.npz', 'test_cache.npz']
    
    if not cache_dir.exists():
        raise FileNotFoundError(f"Cache directory not found: {cache_dir}")
    
    missing = [f for f in required_files if not (cache_dir / f).exists()]
    if missing:
        raise FileNotFoundError(f"Missing cache files: {missing}")
    
    return True


verify_cache_directory(Config)
Config.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration: {Config.MODEL_NAME} v{Config.VERSION}")
print(f"  Input: {Config.IMG_SIZE}x{Config.IMG_SIZE} | Batch: {Config.BATCH_SIZE} | LR: {Config.LEARNING_RATE}")
print(f"  Training: {Config.EPOCHS_STAGE1}+{Config.EPOCHS_STAGE2}+{Config.EPOCHS_STAGE3} epochs")

In [ ]:
# =============================================================================
# Preprocessing Pipeline
# =============================================================================

def apply_clahe(image, clip_limit=2.5, tile_size=(8, 8)):
    """CLAHE contrast enhancement (Rasheed et al.)."""
    if len(image.shape) == 3:
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    else:
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
        return clahe.apply(image)


def apply_sharpening(image):
    """Sharpening filter for tissue structure enhancement (Talukder et al.)."""
    kernel = np.array([[-1, -1, -1], [-1, 9, -1], [-1, -1, -1]], dtype=np.float32)
    sharpened = cv2.filter2D(image, -1, kernel)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def preprocess_mammogram(image, config=Config):
    """
    Complete preprocessing pipeline.
    
    Steps:
    1. Resize to target resolution
    2. Bilateral filter (edge-preserving smoothing)
    3. CLAHE (contrast enhancement)
    4. Sharpening (tissue structure enhancement)
    5. Median filter (noise reduction)
    6. Normalize to [0, 1]
    """
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
    
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.shape[2] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    
    image = cv2.resize(image, (config.IMG_SIZE, config.IMG_SIZE), interpolation=cv2.INTER_LANCZOS4)
    
    if config.USE_BILATERAL:
        image = cv2.bilateralFilter(image, 9, 75, 75)
    
    image = apply_clahe(image, config.CLAHE_CLIP_LIMIT, config.CLAHE_TILE_SIZE)
    
    if config.USE_SHARPENING:
        image = apply_sharpening(image)
    
    if config.USE_MEDIAN_FILTER:
        image = cv2.medianBlur(image, config.MEDIAN_KERNEL)
    
    return image.astype(np.float32) / 255.0

In [ ]:
# =============================================================================
# Data Loading from Preprocessed Cache
# =============================================================================

def load_cached_roi_data(config):
    """
    Load preprocessed ROI data from cache files.
    
    Cache files contain images preprocessed with CLAHE enhancement.
    Expected structure: train_cache.npz, val_cache.npz, test_cache.npz
    Each file contains 'images' and 'labels' arrays.
    
    Args:
        config: Configuration object with CACHE_DIR path
        
    Returns:
        tuple: (X_train, y_train, X_val, y_val, X_test, y_test)
    """
    cache_files = {
        'train': config.CACHE_DIR / 'train_cache.npz',
        'val': config.CACHE_DIR / 'val_cache.npz',
        'test': config.CACHE_DIR / 'test_cache.npz'
    }
    
    # Verify all cache files exist
    for name, path in cache_files.items():
        if not path.exists():
            raise FileNotFoundError(f"Cache file not found: {path}")
    
    print("Loading preprocessed ROI data from cache...")
    
    train_data = np.load(cache_files['train'])
    val_data = np.load(cache_files['val'])
    test_data = np.load(cache_files['test'])
    
    X_train, y_train = train_data['images'], train_data['labels']
    X_val, y_val = val_data['images'], val_data['labels']
    X_test, y_test = test_data['images'], test_data['labels']
    
    # Convert labels to int32 (required for class_weights dict keys)
    y_train = y_train.astype(np.int32)
    y_val = y_val.astype(np.int32)
    y_test = y_test.astype(np.int32)
    
    print(f"  Training:   {X_train.shape[0]} samples, {X_train.nbytes / 1e9:.2f} GB")
    print(f"  Validation: {X_val.shape[0]} samples, {X_val.nbytes / 1e9:.2f} GB")
    print(f"  Test:       {X_test.shape[0]} samples, {X_test.nbytes / 1e9:.2f} GB")
    
    return X_train, y_train, X_val, y_val, X_test, y_test


def resize_images_if_needed(images, target_size):
    """Resize images to target size if dimensions differ."""
    current_size = images.shape[1]
    if current_size == target_size:
        return images
    
    print(f"Resizing images from {current_size}x{current_size} to {target_size}x{target_size}...")
    resized = np.zeros((len(images), target_size, target_size, 3), dtype=np.float32)
    for i, img in enumerate(tqdm(images, desc="Resizing")):
        resized[i] = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LANCZOS4)
    return resized


# Load cached data
X_train, y_train, X_val, y_val, X_test, y_test = load_cached_roi_data(Config)

# Resize if cache was created with different image size
X_train = resize_images_if_needed(X_train, Config.IMG_SIZE)
X_val = resize_images_if_needed(X_val, Config.IMG_SIZE)
X_test = resize_images_if_needed(X_test, Config.IMG_SIZE)

gc.collect()

# Class weights for imbalanced data (keys must be int)
class_counts = Counter(y_train.tolist())
class_weights = {int(k): len(y_train) / (2 * v) for k, v in class_counts.items()}

# Class distribution summary
print("\nClass Distribution:")
for name, labels in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    benign = int((labels == 0).sum())
    malignant = int((labels == 1).sum())
    total = len(labels)
    print(f"  {name}: BENIGN={benign} ({benign/total*100:.1f}%), MALIGNANT={malignant} ({malignant/total*100:.1f}%)")

print(f"\nClass weights: {class_weights}")

In [ ]:
# =============================================================================
# Data Augmentation
# =============================================================================

def create_augmentation_layer(config=Config):
    """Keras augmentation layers based on Talukder et al."""
    return keras.Sequential([
        layers.RandomRotation(config.ROTATION_RANGE / 360, fill_mode='nearest', seed=config.SEED),
        layers.RandomTranslation(config.SHIFT_RANGE, config.SHIFT_RANGE, fill_mode='nearest', seed=config.SEED),
        layers.RandomZoom((-config.ZOOM_RANGE, config.ZOOM_RANGE), fill_mode='nearest', seed=config.SEED),
        layers.RandomFlip('horizontal', seed=config.SEED),
    ], name='augmentation')

In [ ]:
# =============================================================================
# tf.data Pipeline
# =============================================================================

def create_dataset(X, y, config=Config, is_training=True, augment=True):
    """Create tf.data pipeline with caching and prefetching."""
    
    def normalize(image, label):
        mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
        return (image - mean) / std, label
    
    def augment_fn(image, label):
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, 0.1)
        image = tf.image.random_contrast(image, 0.9, 1.1)
        return tf.clip_by_value(image, 0.0, 1.0), label
    
    dataset = tf.data.Dataset.from_tensor_slices((X, y)).cache()
    
    if is_training:
        dataset = dataset.shuffle(min(len(X), 10000), seed=config.SEED, reshuffle_each_iteration=True)
        if augment:
            dataset = dataset.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    
    dataset = dataset.map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(config.BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    
    return dataset


train_dataset = create_dataset(X_train, y_train, Config, is_training=True)
val_dataset = create_dataset(X_val, y_val, Config, is_training=False)
test_dataset = create_dataset(X_test, y_test, Config, is_training=False)

In [ ]:
# =============================================================================
# Model Architecture: DenseNet169 with Block-End Layers
# Reference: Talukder et al. (2025), Rasheed et al. (2025)
# =============================================================================

from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, Activation, 
    Conv2D, GlobalAveragePooling2D, Concatenate
)
from tensorflow.keras.regularizers import l2

# Add missing config attributes
Config.INPUT_SHAPE = (Config.IMG_SIZE, Config.IMG_SIZE, 3)
Config.NUM_CLASSES = 1  # Binary classification (sigmoid output)


def block_end_layer(x, filters, name_prefix, l2_reg=Config.L2_REG, dropout_rate=Config.DROPOUT_GENERAL):
    """
    Block-End Layer (BEL): BN -> ReLU -> Conv(1x1)
    Applied after each dense block to enhance feature discrimination.
    """
    x = BatchNormalization(name=f'{name_prefix}_bn')(x)
    x = Activation('relu', name=f'{name_prefix}_relu')(x)
    x = Conv2D(filters, (1, 1), padding='same', 
               kernel_regularizer=l2(l2_reg), name=f'{name_prefix}_conv')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate, name=f'{name_prefix}_dropout')(x)
    return x


def build_densenet169_bel(input_shape=None, num_classes=None):
    """
    DenseNet169 with Block-End Layers for mammogram classification.
    
    Architecture modifications per literature:
    - BEL after each dense block for enhanced feature learning
    - Progressive dropout rates across blocks
    - L2 regularization for weight decay
    """
    if input_shape is None:
        input_shape = Config.INPUT_SHAPE
    if num_classes is None:
        num_classes = Config.NUM_CLASSES
    
    inputs = Input(shape=input_shape, name='input_layer')
    
    # Load pretrained backbone
    base_model = DenseNet169(weights='imagenet', include_top=False, input_tensor=inputs)
    
    # Dense block output layers
    block_outputs = [
        ('conv2_block6_concat', 256, 0.1),
        ('conv3_block12_concat', 512, 0.15),
        ('conv4_block32_concat', 1024, 0.2),
        ('conv5_block32_concat', 1664, 0.25)
    ]
    
    # Apply BEL after each dense block
    enhanced_features = []
    for layer_name, filters, dropout in block_outputs:
        block_output = base_model.get_layer(layer_name).output
        bel_output = block_end_layer(block_output, filters, f'bel_{layer_name}', 
                                     dropout_rate=dropout)
        pooled = GlobalAveragePooling2D(name=f'gap_{layer_name}')(bel_output)
        enhanced_features.append(pooled)
    
    # Concatenate multi-scale features
    concatenated = Concatenate(name='feature_concat')(enhanced_features)
    
    # Classification head
    x = Dense(512, kernel_regularizer=l2(Config.L2_REG), name='fc1')(concatenated)
    x = BatchNormalization(name='fc1_bn')(x)
    x = Activation('relu', name='fc1_relu')(x)
    x = Dropout(Config.DROPOUT_FINETUNE, name='fc1_dropout')(x)
    
    x = Dense(256, kernel_regularizer=l2(Config.L2_REG), name='fc2')(x)
    x = BatchNormalization(name='fc2_bn')(x)
    x = Activation('relu', name='fc2_relu')(x)
    x = Dropout(Config.DROPOUT_FINETUNE, name='fc2_dropout')(x)
    
    outputs = Dense(num_classes, activation='sigmoid', name='predictions')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='DenseNet169_BEL')
    return model, base_model


model, base_model = build_densenet169_bel()
print(f"Model parameters: {model.count_params():,}")

In [ ]:
# =============================================================================
# Training Callbacks
# =============================================================================

def create_callbacks(config, model_idx=0, stage=1):
    """Create training callbacks for monitoring and checkpointing."""
    callbacks = []
    
    # Best model checkpoint (val_auc)
    callbacks.append(ModelCheckpoint(
        filepath=str(config.CHECKPOINT_DIR / f"model_{model_idx}_stage{stage}_best.keras"),
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    ))
    
    # Global best checkpoint
    callbacks.append(ModelCheckpoint(
        filepath=str(config.CHECKPOINT_DIR / f"model_{model_idx}_global_best.keras"),
        monitor='val_auc', mode='max', save_best_only=True, verbose=0
    ))
    
    # Early stopping
    callbacks.append(EarlyStopping(
        monitor='val_auc', mode='max', patience=8, restore_best_weights=True, verbose=1
    ))
    
    # Learning rate reduction
    callbacks.append(ReduceLROnPlateau(
        monitor='val_auc', mode='max', factor=0.5, patience=4, min_lr=1e-7, verbose=1
    ))
    
    # CSV logger
    callbacks.append(CSVLogger(
        str(config.RESULTS_DIR / f"model_{model_idx}_stage{stage}_history.csv"), append=True
    ))
    
    return callbacks


print("Callbacks: ModelCheckpoint, EarlyStopping (patience=8), ReduceLROnPlateau, CSVLogger")

In [ ]:
# =============================================================================
# Three-Stage Progressive Training
# =============================================================================

def freeze_base_model(base_model):
    """Freeze all layers in base model."""
    for layer in base_model.layers:
        layer.trainable = False


def unfreeze_layers(base_model, num_layers):
    """Unfreeze last N layers, keeping BatchNorm frozen."""
    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-num_layers:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True


def compile_model(model, config, learning_rate):
    """Compile model with label smoothing loss."""
    model.compile(
        optimizer=Adamax(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=config.LABEL_SMOOTHING),
        metrics=[keras.metrics.AUC(name='auc'), 'accuracy']
    )


def train_model_progressive(model, base_model, train_ds, val_ds, config, 
                            model_idx=0, class_weights=None):
    """
    Three-stage progressive training strategy.
    
    Stage 1: Classifier warm-up (frozen base)
    Stage 2: Progressive unfreezing (40% of base)
    Stage 3: Fine-tuning (70% of base)
    """
    all_history = {'stage1': None, 'stage2': None, 'stage3': None}
    num_layers = len(base_model.layers)
    
    # Stage 1: Classifier warm-up
    print(f"\n{'='*60}\nStage 1: Classifier Warm-up (Model {model_idx})\n{'='*60}")
    freeze_base_model(base_model)
    compile_model(model, config, config.LEARNING_RATE)
    
    history1 = model.fit(
        train_ds, validation_data=val_ds, epochs=config.EPOCHS_STAGE1,
        callbacks=create_callbacks(config, model_idx, stage=1),
        class_weight=class_weights, verbose=1
    )
    all_history['stage1'] = history1.history
    
    # Stage 2: Progressive unfreezing (40%)
    print(f"\n{'='*60}\nStage 2: Progressive Unfreezing (Model {model_idx})\n{'='*60}")
    unfreeze_count = int(num_layers * 0.4)
    unfreeze_layers(base_model, unfreeze_count)
    compile_model(model, config, config.LEARNING_RATE / 3)
    
    history2 = model.fit(
        train_ds, validation_data=val_ds, epochs=config.EPOCHS_STAGE2,
        callbacks=create_callbacks(config, model_idx, stage=2),
        class_weight=class_weights, verbose=1
    )
    all_history['stage2'] = history2.history
    
    # Stage 3: Fine-tuning (70%)
    print(f"\n{'='*60}\nStage 3: Fine-tuning (Model {model_idx})\n{'='*60}")
    unfreeze_count = int(num_layers * 0.7)
    unfreeze_layers(base_model, unfreeze_count)
    compile_model(model, config, config.LEARNING_RATE / 10)
    
    history3 = model.fit(
        train_ds, validation_data=val_ds, epochs=config.EPOCHS_STAGE3,
        callbacks=create_callbacks(config, model_idx, stage=3),
        class_weight=class_weights, verbose=1
    )
    all_history['stage3'] = history3.history
    
    return all_history


print(f"Progressive training: {Config.EPOCHS_STAGE1}+{Config.EPOCHS_STAGE2}+{Config.EPOCHS_STAGE3} epochs")

In [ ]:
# =============================================================================
# Ensemble Training with Checkpoint Resumption
# =============================================================================

def check_existing_checkpoints(config):
    """Check for existing model checkpoints."""
    existing = {}
    for i in range(config.NUM_ENSEMBLE):
        global_best = config.CHECKPOINT_DIR / f"model_{i}_global_best.keras"
        stage3_best = config.CHECKPOINT_DIR / f"model_{i}_stage3_best.keras"
        if global_best.exists():
            existing[i] = global_best
        elif stage3_best.exists():
            existing[i] = stage3_best
    return existing


def create_densenet169_model(config, model_idx=0):
    """Create fresh DenseNet169 model with seed variation."""
    tf.random.set_seed(config.SEED + model_idx * 100)
    return build_densenet169_bel()


def train_ensemble(config, train_ds, val_ds, class_weights=None,
                   resume_training=False, skip_trained_models=True):
    """Train ensemble of DenseNet169 models."""
    ensemble_histories = []
    existing_checkpoints = check_existing_checkpoints(config)
    
    print(f"\nTraining Ensemble: {config.NUM_ENSEMBLE} models")
    
    if existing_checkpoints:
        print(f"Found {len(existing_checkpoints)} existing checkpoints")
    
    for i in range(config.NUM_ENSEMBLE):
        print(f"\n{'='*60}\nModel {i+1}/{config.NUM_ENSEMBLE}\n{'='*60}")
        
        if skip_trained_models and i in existing_checkpoints:
            print(f"Skipping model {i} - checkpoint exists")
            ensemble_histories.append({'stage1': None, 'stage2': None, 'stage3': None})
            continue
        
        model, base_model = create_densenet169_model(config, model_idx=i)
        
        if resume_training and i in existing_checkpoints:
            print(f"Loading checkpoint: {existing_checkpoints[i].name}")
            try:
                model.load_weights(str(existing_checkpoints[i]))
            except Exception as e:
                print(f"Could not load weights: {e}")
        
        history = train_model_progressive(
            model=model, base_model=base_model,
            train_ds=train_ds, val_ds=val_ds,
            config=config, model_idx=i, class_weights=class_weights
        )
        ensemble_histories.append(history)
        
        # Save history
        history_path = config.RESULTS_DIR / f"model_{i}_full_history.json"
        serializable = {stage: {k: [float(v) for v in vals] for k, vals in h.items()} 
                       for stage, h in history.items() if h}
        with open(history_path, 'w') as f:
            json.dump(serializable, f, indent=2)
        
        del model, base_model
        gc.collect()
        tf.keras.backend.clear_session()
    
    print(f"\nEnsemble training complete")
    return ensemble_histories


# Training control
SKIP_TRAINED_MODELS = True
RESUME_TRAINING = False

existing = check_existing_checkpoints(Config)
print(f"Existing checkpoints: {len(existing)}")
print(f"Total epochs per model: {Config.EPOCHS_STAGE1 + Config.EPOCHS_STAGE2 + Config.EPOCHS_STAGE3}")

ensemble_histories = train_ensemble(
    config=Config, train_ds=train_dataset, val_ds=val_dataset,
    class_weights=class_weights, resume_training=RESUME_TRAINING,
    skip_trained_models=SKIP_TRAINED_MODELS
)

In [ ]:
# =============================================================================
# Test-Time Augmentation (TTA)
# =============================================================================

def apply_tta(image, num_augmentations=8):
    """
    Apply test-time augmentation to single image.
    Augmentations: original, horizontal flip, rotations (-10,-5,+5,+10), zoom (95%,105%)
    """
    h, w = image.shape[:2]
    augmented = [image, np.fliplr(image).copy()]
    
    # Rotations
    for angle in [-5, 5, -10, 10]:
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        augmented.append(cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REPLICATE))
    
    # Zoom variations
    for zoom in [0.95, 1.05]:
        new_h, new_w = int(h * zoom), int(w * zoom)
        if zoom < 1:
            zoomed = cv2.resize(image, (new_w, new_h))
            pad_top, pad_left = (h - new_h) // 2, (w - new_w) // 2
            padded = np.pad(zoomed, ((pad_top, h-new_h-pad_top), (pad_left, w-new_w-pad_left), (0,0)), mode='edge')
            augmented.append(padded)
        else:
            zoomed = cv2.resize(image, (new_w, new_h))
            start_y, start_x = (new_h - h) // 2, (new_w - w) // 2
            augmented.append(zoomed[start_y:start_y+h, start_x:start_x+w])
    
    return augmented[:num_augmentations]


def predict_with_tta(model, image, num_augmentations=8):
    """Predict with TTA and average results."""
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    augmented = apply_tta(image, num_augmentations)
    batch = np.stack(augmented, axis=0)
    batch_normalized = (batch - mean) / std
    
    predictions = model.predict(batch_normalized, verbose=0, batch_size=len(augmented))
    return float(np.mean(predictions))


def ensemble_predict_with_tta(models, image, num_augmentations=8):
    """Ensemble prediction with TTA."""
    return np.mean([predict_with_tta(m, image, num_augmentations) for m in models])


print(f"TTA: {Config.TTA_AUGMENTATIONS} augmentations per image")

In [ ]:
# =============================================================================
# Model Evaluation
# =============================================================================

def load_ensemble_models(config):
    """Load best checkpoint for each ensemble model."""
    models = []
    for i in range(config.NUM_ENSEMBLE):
        checkpoint_path = config.CHECKPOINT_DIR / f"model_{i}_global_best.keras"
        if checkpoint_path.exists():
            models.append(keras.models.load_model(str(checkpoint_path)))
    print(f"Loaded {len(models)} models")
    return models


def evaluate_ensemble(models, X_test, y_test, config, use_tta=True):
    """Evaluate ensemble on test set."""
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    all_predictions = []
    individual_aucs = []
    
    for i, model in enumerate(models):
        if use_tta:
            preds = np.array([predict_with_tta(model, img, config.TTA_AUGMENTATIONS) 
                             for img in tqdm(X_test, desc=f"Model {i}")])
        else:
            X_norm = (X_test - mean) / std
            preds = model.predict(X_norm, verbose=0).flatten()
        
        all_predictions.append(preds)
        auc = roc_auc_score(y_test, preds)
        individual_aucs.append(auc)
        print(f"Model {i} AUC: {auc:.4f}")
    
    # Ensemble predictions
    ensemble_preds = np.mean(all_predictions, axis=0)
    ensemble_binary = (ensemble_preds > 0.5).astype(int)
    
    # Metrics
    ensemble_auc = roc_auc_score(y_test, ensemble_preds)
    accuracy = accuracy_score(y_test, ensemble_binary)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, ensemble_binary, average='binary')
    cm = confusion_matrix(y_test, ensemble_binary)
    tn, fp, fn, tp = cm.ravel()
    
    results = {
        'ensemble_auc': ensemble_auc, 'accuracy': accuracy,
        'precision': precision, 'recall': recall, 'f1_score': f1,
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'individual_aucs': individual_aucs, 'confusion_matrix': cm.tolist(),
        'predictions': ensemble_preds.tolist(), 'true_labels': y_test.tolist()
    }
    
    print(f"\nEnsemble AUC: {ensemble_auc:.4f} | Accuracy: {accuracy:.4f}")
    print(f"Sensitivity: {results['sensitivity']:.4f} | Specificity: {results['specificity']:.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    
    return results


# Load and evaluate
ensemble_models = load_ensemble_models(Config)
if ensemble_models:
    results = evaluate_ensemble(ensemble_models, X_test, y_test, Config, use_tta=True)
else:
    print("No models found for evaluation")

In [ ]:
# =============================================================================
# Training History Visualization
# =============================================================================

def plot_training_history(histories, config):
    """Plot training curves for ensemble models."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    colors = plt.cm.Set2(np.linspace(0, 1, len(histories)))
    
    for idx, history in enumerate(histories):
        loss, val_loss, auc, val_auc = [], [], [], []
        for stage in ['stage1', 'stage2', 'stage3']:
            if history[stage]:
                loss.extend(history[stage].get('loss', []))
                val_loss.extend(history[stage].get('val_loss', []))
                auc.extend(history[stage].get('auc', []))
                val_auc.extend(history[stage].get('val_auc', []))
        
        epochs = range(1, len(loss) + 1)
        axes[0, 0].plot(epochs, loss, color=colors[idx], linestyle='-', label=f'M{idx} train')
        axes[0, 0].plot(epochs, val_loss, color=colors[idx], linestyle='--', label=f'M{idx} val')
        axes[0, 1].plot(epochs, auc, color=colors[idx], linestyle='-', label=f'M{idx} train')
        axes[0, 1].plot(epochs, val_auc, color=colors[idx], linestyle='--', label=f'M{idx} val')
    
    # Stage boundaries
    s1, s2 = config.EPOCHS_STAGE1, config.EPOCHS_STAGE1 + config.EPOCHS_STAGE2
    for ax in axes[0]:
        ax.axvline(x=s1, color='gray', linestyle=':', alpha=0.7)
        ax.axvline(x=s2, color='gray', linestyle=':', alpha=0.7)
    
    axes[0, 0].set(xlabel='Epoch', ylabel='Loss', title='Training Loss')
    axes[0, 0].legend(fontsize=8); axes[0, 0].grid(True, alpha=0.3)
    axes[0, 1].set(xlabel='Epoch', ylabel='AUC', title='Training AUC')
    axes[0, 1].legend(fontsize=8); axes[0, 1].grid(True, alpha=0.3)
    
    # Best AUC comparison
    best_aucs = [max([max(h[s].get('val_auc', [0])) for s in ['stage1','stage2','stage3'] if h[s]], default=0) 
                 for h in histories]
    axes[1, 0].bar(range(len(best_aucs)), best_aucs, color=colors[:len(best_aucs)])
    axes[1, 0].set(xlabel='Model', ylabel='Best Val AUC', title='Best AUC per Model')
    for i, v in enumerate(best_aucs):
        axes[1, 0].text(i, v + 0.005, f'{v:.4f}', ha='center')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Configuration
    cfg_text = f"Input: {config.IMG_SIZE}x{config.IMG_SIZE}\nLR: {config.LEARNING_RATE}\nBatch: {config.BATCH_SIZE}\nStages: {config.EPOCHS_STAGE1}+{config.EPOCHS_STAGE2}+{config.EPOCHS_STAGE3}"
    axes[1, 1].text(0.1, 0.5, cfg_text, transform=axes[1, 1].transAxes, fontsize=12, 
                   fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))
    axes[1, 1].axis('off'); axes[1, 1].set_title('Configuration')
    
    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / "training_history.png", dpi=150, bbox_inches='tight')
    plt.show()


if 'ensemble_histories' in dir() and ensemble_histories:
    plot_training_history(ensemble_histories, Config)
else:
    print("No training history available (training was skipped or not yet run)")

In [ ]:
# =============================================================================
# ROC Curve and Confusion Matrix
# =============================================================================

def plot_evaluation_results(results, config):
    """Plot ROC curve and confusion matrix."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    y_true = np.array(results['true_labels'])
    y_pred = np.array(results['predictions'])
    
    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    auc_score = results['ensemble_auc']
    
    axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {auc_score:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[0].fill_between(fpr, tpr, alpha=0.2)
    
    # Optimal threshold (Youden's J)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    axes[0].plot(fpr[optimal_idx], tpr[optimal_idx], 'ro', markersize=10, 
                label=f'Optimal (t={optimal_threshold:.3f})')
    
    axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate', 
               title='ROC Curve - DenseNet169 Ensemble')
    axes[0].legend(loc='lower right'); axes[0].grid(True, alpha=0.3)
    
    # Confusion Matrix
    cm = np.array(results['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
               xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES, 
               ax=axes[1], annot_kws={'size': 16})
    axes[1].set(xlabel='Predicted', ylabel='Actual', title='Confusion Matrix')
    
    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / "roc_confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    return optimal_threshold


if 'results' in dir():
    optimal_threshold = plot_evaluation_results(results, Config)
    print(f"Optimal threshold: {optimal_threshold:.4f}")

In [ ]:
# =============================================================================
# Save Results
# =============================================================================

def save_results(results, config):
    """Save final results to JSON."""
    final_results = {
        'model_name': config.MODEL_NAME,
        'version': config.VERSION,
        'timestamp': datetime.now().strftime("%Y%m%d_%H%M%S"),
        'configuration': {
            'img_size': config.IMG_SIZE, 'batch_size': config.BATCH_SIZE,
            'learning_rate': config.LEARNING_RATE, 'optimizer': config.OPTIMIZER,
            'epochs': f"{config.EPOCHS_STAGE1}+{config.EPOCHS_STAGE2}+{config.EPOCHS_STAGE3}",
            'num_ensemble': config.NUM_ENSEMBLE, 'tta_augmentations': config.TTA_AUGMENTATIONS
        },
        'metrics': {
            'ensemble_auc': results['ensemble_auc'], 'accuracy': results['accuracy'],
            'precision': results['precision'], 'recall': results['recall'],
            'f1_score': results['f1_score'], 'sensitivity': results['sensitivity'],
            'specificity': results['specificity'], 'individual_aucs': results['individual_aucs']
        },
        'confusion_matrix': results['confusion_matrix'],
        'dataset': {'train': len(X_train), 'val': len(X_val), 'test': len(X_test)}
    }
    
    results_path = config.RESULTS_DIR / "densenet169_v1_results.json"
    with open(results_path, 'w') as f:
        json.dump(final_results, f, indent=2)
    
    print(f"Results saved: {results_path}")
    print(f"\nFinal Metrics:")
    print(f"  AUC: {results['ensemble_auc']:.4f}")
    print(f"  Accuracy: {results['accuracy']:.4f}")
    print(f"  Sensitivity: {results['sensitivity']:.4f}")
    print(f"  Specificity: {results['specificity']:.4f}")
    
    return final_results


if 'results' in dir():
    final_results = save_results(results, Config)

In [ ]:
# =============================================================================
# Grad-CAM++ Visualization (Explainability)
# =============================================================================

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    """Generate Grad-CAM++ heatmap for model interpretation."""
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if isinstance(layer, (layers.Conv2D, layers.Activation)):
                last_conv_layer_name = layer.name
                break
    
    grad_model = Model(inputs=model.inputs,
                       outputs=[model.get_layer(last_conv_layer_name).output, model.output])
    
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_channel = predictions[:, 0] if len(predictions.shape) > 1 else predictions
    
    grads = tape.gradient(class_channel, conv_outputs)
    
    # Grad-CAM++ weights
    grads_power_2, grads_power_3 = grads ** 2, grads ** 3
    sum_activations = tf.reduce_sum(conv_outputs, axis=(1, 2))
    eps = 1e-7
    
    aij = grads_power_2 / (2 * grads_power_2 + sum_activations[:, tf.newaxis, tf.newaxis, :] * grads_power_3 + eps)
    weights = tf.reduce_sum(aij * tf.nn.relu(grads), axis=(1, 2))
    
    heatmap = tf.reduce_sum(weights[:, tf.newaxis, tf.newaxis, :] * conv_outputs, axis=-1)
    heatmap = tf.nn.relu(heatmap) / (tf.reduce_max(heatmap) + eps)
    
    return heatmap[0].numpy()


def visualize_gradcam(model, images, labels, config, num_samples=6):
    """Visualize Grad-CAM++ for sample images."""
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 3))
    indices = np.random.choice(len(images), num_samples, replace=False)
    
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    for i, idx in enumerate(indices):
        img, label = images[idx], labels[idx]
        img_batch = np.expand_dims((img - mean) / std, 0)
        
        pred = model.predict(img_batch, verbose=0)[0, 0]
        pred_class = 'MALIGNANT' if pred > 0.5 else 'BENIGN'
        true_class = config.CLASS_NAMES[label]
        
        try:
            heatmap = make_gradcam_heatmap(img_batch, model)
            heatmap_resized = cv2.resize(heatmap, (config.IMG_SIZE, config.IMG_SIZE))
        except:
            heatmap_resized = np.zeros((config.IMG_SIZE, config.IMG_SIZE))
        
        axes[i, 0].imshow(img); axes[i, 0].set_title(f'True: {true_class}'); axes[i, 0].axis('off')
        axes[i, 1].imshow(heatmap_resized, cmap='jet'); axes[i, 1].set_title(f'Pred: {pred_class} ({pred:.2f})'); axes[i, 1].axis('off')
        
        heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
        overlay = np.uint8(img * 255 * 0.6 + heatmap_colored * 0.4)
        
        correct = pred_class == true_class
        axes[i, 2].imshow(overlay); axes[i, 2].set_title('Correct' if correct else 'Incorrect', color='green' if correct else 'red'); axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(config.RESULTS_DIR / "gradcam_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()


if ensemble_models:
    visualize_gradcam(ensemble_models[0], X_test, y_test, Config, num_samples=6)

In [ ]:
# =============================================================================
# Summary
# =============================================================================

print(f"""
DenseNet169 V1 Training Complete
================================

Architecture:
  - DenseNet169 with Block-End Layers (BEL)
  - Input: {Config.IMG_SIZE}x{Config.IMG_SIZE}x3
  - Optimizer: Adamax (LR={Config.LEARNING_RATE})

Preprocessing:
  - CLAHE (clip={Config.CLAHE_CLIP_LIMIT})
  - Sharpening + Bilateral + Median filter

Training:
  - 3-stage progressive: {Config.EPOCHS_STAGE1}+{Config.EPOCHS_STAGE2}+{Config.EPOCHS_STAGE3} epochs
  - Ensemble: {Config.NUM_ENSEMBLE} models
  - TTA: {Config.TTA_AUGMENTATIONS} augmentations

Output Files:
  - Checkpoints: {Config.CHECKPOINT_DIR}
  - Results: {Config.RESULTS_DIR}

References:
  1. Talukder et al. (2025). Results in Engineering.
  2. Rasheed et al. (2025). Discover Oncology.

Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
""")